In [1]:
import kagglehub
path = kagglehub.dataset_download("iamsouravbanerjee/customer-shopping-trends-dataset")

100%|██████████| 146k/146k [00:00<00:00, 381kB/s]

Extracting files...


In [2]:
import os
import pandas as pd

# List contents of the downloaded path to find the dataset file(s)
print(f"Contents of the downloaded path: {os.listdir(path)}")

Contents of the downloaded path: ['shopping_trends_updated.csv', 'shopping_trends.csv']


Please examine the output from the previous cell and tell me the exact filename of the CSV file. For example, if the output shows `['customer_shopping_trends.csv']`, then the filename is `customer_shopping_trends.csv`.

In [3]:
file_name = 'shopping_trends_updated.csv'
df = pd.read_csv(os.path.join(path, file_name))
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [4]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3900 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3900.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.749949,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716223,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.700000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Define target and features
X = df.drop(['Customer ID', 'Purchase Amount (USD)'], axis=1)
y = df['Purchase Amount (USD)']

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create a pipeline with preprocessing and a placeholder for the model
# We'll add the model later after the data is transformed and split
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Apply preprocessing to the features
X_processed = pipeline.fit_transform(X)

# Get feature names after one-hot encoding for categorical features
# This helps in understanding the columns of the processed data
cat_feature_names = pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = list(numerical_features) + list(cat_feature_names)

print(f"Original features: {X.columns.tolist()}")
print(f"Numerical features: {numerical_features.tolist()}")
print(f"Categorical features: {categorical_features.tolist()}")
print(f"Shape of preprocessed features: {X_processed.shape}")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

Original features: ['Age', 'Gender', 'Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']
Numerical features: ['Age', 'Review Rating', 'Previous Purchases']
Categorical features: ['Gender', 'Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Payment Method', 'Frequency of Purchases']
Shape of preprocessed features: (3900, 142)
X_train shape: (3120, 142), y_train shape: (3120,)
X_test shape: (780, 142), y_test shape: (780,)


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define the deep learning model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1) # Output layer for regression
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])

# Display model summary
model.summary()

# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=50, # You can adjust the number of epochs
    batch_size=32,
    validation_split=0.2, # Use a portion of training data for validation
    verbose=1
)

# Evaluate the model on the test set
loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"\nModel Evaluation on Test Set:")
print(f"Mean Squared Error: {loss:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        18,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,673 (112.00 KB)

 Trainable params: 28,673 (112.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
78/78 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 2185.4939 - mean_absolute_error: 38.8601 - val_loss: 558.0839 - val_mean_absolute_error: 20.4211
Epoch 2/50
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 565.6393 - mean_absolute_error: 20.5595 - val_loss: 547.0080 - val_mean_absolute_error: 20.3189
Epoch 3/50
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 551.4605 - mean_absolute_error: 20.2501 - val_loss: 550.5673 - val_mean_absolute_error: 20.3400
Epoch 4/50
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 549.5687 - mean_absolute_error: 20.1620 - val_loss: 565.3888 - val_mean_absolute_error: 20.4977
Epoch 5/50
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 542.5425 - mean_absolute_error: 19.9832 - val_loss: 558.9481 - val_mean_absolute_error: 20.4228
Epoch 6/50
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 534.6943 - mean_absolute_error: 19.8634 - val_loss: 559.3962 - val_mean_absolute_error: 20.3864
Epoch 7/50
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 528.3336 - mean_

In [7]:
# Install Streamlit
!pip install streamlit joblib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 54.0 MB/s eta 0:00:00


In [8]:
import joblib
import tensorflow as tf

# Save the preprocessor pipeline
joblib.dump(pipeline, 'preprocessor.joblib')

# Save the Keras model
model.save('deep_learning_model.keras')

print("Preprocessor and model saved successfully.")

Preprocessor and model saved successfully.


In [21]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import os

# Define the path to the dataset - this needs to be accessible from within the app
DATASET_PATH = '/root/.cache/kagglehub/datasets/iamsouravbanerjee/customer-shopping-trends-dataset/versions/2/shopping_trends_updated.csv'

# Load the preprocessor and model
preprocessor_pipeline = joblib.load('preprocessor.joblib')
deep_learning_model = tf.keras.models.load_model('deep_learning_model.keras')

# Load the original dataframe to get feature names and unique categorical values
df_original = pd.read_csv(DATASET_PATH)

# Define X and original_features from the reloaded dataframe
X_original = df_original.drop(['Customer ID', 'Purchase Amount (USD)'], axis=1)
original_features = X_original.columns.tolist()

# --- Main App Content ---
st.title('🛒 Customer Purchase Amount Predictor 💰')
st.write('Welcome! Use the sidebar to enter customer details and predict their shopping spend.')

# --- Sidebar for Input ---
st.sidebar.title('📝 Enter Customer Details')
st.sidebar.write('Adjust the parameters below to get a purchase amount prediction.')

input_data = {}

# Numerical features in sidebar
numerical_features = ['Age', 'Review Rating', 'Previous Purchases']
for feature in numerical_features:
    if feature == 'Review Rating':
        input_data[feature] = st.sidebar.slider(f'Select {feature}', 1.0, 5.0, 3.5, 0.1)
    elif feature == 'Age':
        input_data[feature] = st.sidebar.slider(f'Select {feature}', 18, 70, 30)
    else: # Previous Purchases
        input_data[feature] = st.sidebar.number_input(f'Enter {feature}', min_value=0, value=10)

# Categorical features in sidebar
categorical_features = [f for f in original_features if f not in numerical_features]

for feature in categorical_features:
    unique_vals = df_original[feature].unique().tolist()
    input_data[feature] = st.sidebar.selectbox(f'Select {feature}', unique_vals)

# Predict button in the main area for better visibility
if st.button('🚀 Predict Purchase Amount'):
    # Create a DataFrame from input data
    input_df = pd.DataFrame([input_data])

    # Create a dummy DataFrame with all original features and then fill it with user input
    # This is crucial for `ColumnTransformer` to work correctly and maintain feature order
    dummy_df = pd.DataFrame(columns=original_features)
    dummy_df = pd.concat([dummy_df, input_df], ignore_index=True)

    # Ensure input_df has all original_features, in the correct order
    processed_input = preprocessor_pipeline.named_steps['preprocessor'].transform(dummy_df[original_features])

    # Make prediction
    prediction = deep_learning_model.predict(processed_input)[0][0]

    st.success(f'### Predicted Purchase Amount (USD): ${prediction:.2f} 💸')

st.markdown("---")
st.markdown("💡 *This model predicts potential customer spending based on various shopping trends attributes.*")

Overwriting app.py


To run the Streamlit app, execute the next cell. After running, a link will appear, usually resembling `your_ip_address:8501`. Click on this link to open the app in a new tab.

In [12]:
!pip install --ignore-installed blinker
!pip install streamlit

In [13]:
!pip install streamlit pyngrok

In [14]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [22]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [23]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run app.py &


2026-08-20 12:51:19.730 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.137.107:8501

2026-08-20 12:51:30.611301: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/content/app.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dummy_df = pd.concat([dummy_df, input_df], ignore_index=True)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
  Stopping...


In [10]:
!streamlit run app.py --server.port 8501

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py
